In [76]:
import pandas as pd
import numpy as np
import seaborn as sns
import xarray as xr
import scipy.stats as stats
import matplotlib.pyplot as plt
import tqdm as tqdm
import sklearn 

from scipy.integrate import simpson
from sklearn.preprocessing import StandardScaler

In [77]:
SM_data_all = pd.read_csv('/data/keeling/a/cstraus2/Phd_research/T_Ford_7-10-2024.txt')
#import missing data files
cmi_data = pd.read_csv('/data/keeling/a/cstraus2/Phd_research/T_Ford_CMI.csv')
mon_data = pd.read_csv('/data/keeling/a/cstraus2/Phd_research/T_Ford_MON.csv')
siu_data = pd.read_csv('/data/keeling/a/cstraus2/Phd_research/T_Ford_SIU.csv')
#set station ID to be one ID, currently has two one with and one without spaces
cmi_data['Station']= 'cmi'
mon_data['Station'] = 'mon'
siu_data['Station'] = 'siu'

#change column names to match, have 'Precip' and 'preciptation'
cmi_data['Precipitation (in)'] = cmi_data['Precip (in)']
mon_data['Precipitation (in)'] = mon_data['Precip (in)']
siu_data['Precipitation (in)'] = siu_data['Precip (in)']

cmi_data['Date & Time (CST)'] = pd.to_datetime(cmi_data['Date & Time (CST)'])
mon_data['Date & Time (CST)'] = pd.to_datetime(mon_data['Date & Time (CST)'])
siu_data['Date & Time (CST)'] = pd.to_datetime(siu_data['Date & Time (CST)'])
SM_data_all['Date & Time (CST)'] = pd.to_datetime(SM_data_all['Date & Time (CST)'].str[:19])
#concatenate all data
SM_data_all = pd.concat([SM_data_all, cmi_data, mon_data, siu_data], axis = 0)
SM_data_all = SM_data_all.reset_index(drop = True)
SM_data_all = SM_data_all.drop('Precip (in)', axis = 1)
#drop 2008 data because they switched from bucket weight units to inches per hour, data complete mess.
SM_data_all['Year'] = SM_data_all['Date & Time (CST)'].dt.year
SM_data_all = SM_data_all[SM_data_all['Year'] != 2008]
SM_data_all = SM_data_all.drop('Year', axis = 1)

What is wfv?
water fraction volume cm3/cm3 volumetric fraction of soil column that is water (relative to total 
absolut volume)
metadata:
https://www.isws.illinois.edu/warm/climnet/stnmetadata.asp


# Data is not in order and soil moisture values =<0 and >0.7 need to be removed, replace with NaN

we are assuming no data means there is no rain since we cannot measure ammount and precipitation and
we are only concerned with the peak values and difference and can assume cutting some events short 
or not including them has low impact on the peak values and changes

## remove rain that is over 3 in/hr rate, these are unlikely to be true for Illinois and only occur a handful of times in the entire datasets
## set unprobable threshold to remove for isolate rainrates, ie if precip is over 2 in/hr but there is no precip for timestep before and after it is unlikely to be a real measurement 

In [78]:
#clean data, remove impossible values
def clean_data(data,precip_cutoff, sm_cutoff, unlikely_threshold):
    Data_column_names = ['Soil Moisture 5cm (wfv)', 'Soil Moisture 10cm (wfv)', 
                     'Soil Moisture 20cm (wfv)','Soil Moisture 50cm (wfv)', 
                     'Soil Moisture 100cm (wfv)', 'Soil Moisture 150cm (wfv)']
    for i in range(len(Data_column_names)):
        data.loc[data[Data_column_names[i]] <= 0, Data_column_names[i]] = np.NaN 
        data.loc[data[Data_column_names[i]] >sm_cutoff, Data_column_names[i]] = np.NaN

    data.loc[data['Precipitation (in)'] > precip_cutoff, 'Precipitation (in)'] =np.NaN
    data['Precipitation_diff'] = np.NaN
    data['Precipitation_downshift'] = np.NaN
    data['Precipitation_upshift'] = np.NaN
    
    #station's data are not linear, but scrabmled, put in order
    Stations = list(data.Station.unique())
    first_station = Stations[0]
    for s in Stations:
        station_data = data.loc[data['Station'] == s]

        station_data = station_data.sort_values(by = 'Date & Time (CST)')
        station_data = station_data.reset_index(drop = True)

        station_data['Precipitation_diff'] = station_data['Precipitation (in)'].diff()
        station_data['Precipitation_downshift'] = station_data['Precipitation (in)'].shift(1)
        station_data[ 'Precipitation_upshift'] = station_data['Precipitation (in)'].shift(-1)

        if s == first_station:
            new_df = station_data
        else:
            new_df = pd.concat([new_df, station_data], axis = 0)

    new_df = new_df.reset_index(drop = True)
    #switch unlikely events with Nan
    new_df.loc[(new_df['Precipitation_diff'] >= unlikely_threshold) &  
                                 (new_df['Precipitation_upshift'] == 0), 
                                 'Precipitation (in)' ] = np.NaN
    return new_df


In [79]:
sm_max_threshold = 0.7
precip_rate_max_threshold = 4
unlikely_isolated_precip = 2
SM_data_cleaned = clean_data(SM_data_all, precip_cutoff=precip_rate_max_threshold, 
                                          sm_cutoff= sm_max_threshold, 
                                          unlikely_threshold= unlikely_isolated_precip)

In [7]:
#visualize cleaned and sorted data
"""""
Stations = SM_data_cleaned.Station.unique()
Data_column_names = ['Precipitation (in)', 'Soil Moisture 5cm (wfv)', 'Soil Moisture 10cm (wfv)', 
                     'Soil Moisture 20cm (wfv)','Soil Moisture 50cm (wfv)', 
                     'Soil Moisture 100cm (wfv)', 'Soil Moisture 150cm (wfv)']

for S in Stations:
    station_data = SM_data_cleaned.loc[SM_data_cleaned['Station'] == S]
    dtv = station_data['Date & Time (CST)'][:]
    precip_diff = station_data['Precipitation (in)'].diff()

    print('max of precip values', station_data['Precipitation (in)'].max())
    print('min of sm values', station_data['Soil Moisture 5cm (wfv)'].min())
    print('max of sm values', station_data['Soil Moisture 5cm (wfv)'].max())
    fig1, ax1 = plt.subplots(nrows = 4, ncols = 1, figsize = (15,8))

    ax1[0].plot(dtv,station_data['Precipitation (in)'], label = 'Precipitation')
    ax1[1].plot(dtv, station_data['Soil Moisture 5cm (wfv)'], label = 'SM 5cm')
    ax1[1].plot(dtv, station_data['Soil Moisture 10cm (wfv)'], label = 'SM 10cm')
    ax1[1].plot(dtv, station_data['Soil Moisture 20cm (wfv)'], label = 'SM 20cm')
    ax1[1].plot(dtv, station_data['Soil Moisture 50cm (wfv)'], label = 'SM 50cm')
    ax1[1].plot(dtv, station_data['Soil Moisture 100cm (wfv)'], label = 'SM 100cm')
    ax1[1].plot(dtv, station_data['Soil Moisture 150cm (wfv)'], label = 'SM 150cm')
    ax1[1].legend()

    ax1[2].plot(precip_diff)

    ax1[3].plot(dtv, station_data['Soil Moisture 5cm (wfv)'].diff(), label = 'SM 5cm diff')
    ax1[3].plot(dtv, station_data['Soil Moisture 10cm (wfv)'].diff(), label = 'SM 10cm diff')
    ax1[3].plot(dtv, station_data['Soil Moisture 20cm (wfv)'].diff(), label = 'SM 20cm diff')
    ax1[3].plot(dtv, station_data['Soil Moisture 50cm (wfv)'].diff(), label = 'SM 50cm diff')
    ax1[3].plot(dtv, station_data['Soil Moisture 100cm (wfv)'].diff(), label = 'SM 100cm diff')
    ax1[3].plot(dtv, station_data['Soil Moisture 150cm (wfv)'].diff(), label = 'SM 150cm diff')
    ax1[3].legend()

    ax1[0].set_title('Station %s' %S)
    ax1[1].set_title('Station %s' %S)
    ax1[2].set_title('Precip rate change for %s' %S)
    ax1[3].set_title('SM differences for %s' %S)
    plt.tight_layout()
    plt.show()
"""

'""\nStations = SM_data_cleaned.Station.unique()\nData_column_names = [\'Precipitation (in)\', \'Soil Moisture 5cm (wfv)\', \'Soil Moisture 10cm (wfv)\', \n                     \'Soil Moisture 20cm (wfv)\',\'Soil Moisture 50cm (wfv)\', \n                     \'Soil Moisture 100cm (wfv)\', \'Soil Moisture 150cm (wfv)\']\n\nfor S in Stations:\n    station_data = SM_data_cleaned.loc[SM_data_cleaned[\'Station\'] == S]\n    dtv = station_data[\'Date & Time (CST)\'][:]\n    precip_diff = station_data[\'Precipitation (in)\'].diff()\n\n    print(\'max of precip values\', station_data[\'Precipitation (in)\'].max())\n    print(\'min of sm values\', station_data[\'Soil Moisture 5cm (wfv)\'].min())\n    print(\'max of sm values\', station_data[\'Soil Moisture 5cm (wfv)\'].max())\n    fig1, ax1 = plt.subplots(nrows = 4, ncols = 1, figsize = (15,8))\n\n    ax1[0].plot(dtv,station_data[\'Precipitation (in)\'], label = \'Precipitation\')\n    ax1[1].plot(dtv, station_data[\'Soil Moisture 5cm (wfv)\'],

In [80]:
Column_sum_depth = [10, 100]
sensor_depths = np.array([5, 10, 20, 50, 100, 150])

#Column_depth is Column_sum_depth, and depths is sensor_depths
def column_weighted_sm(data, Column_depth, depths):
    for m in range(len(Column_depth)):
        n = np.where(sensor_depths == Column_depth[m])[0][0]
        depth_weights = np.zeros(len(depths))
        Weighted_column_names = str(Column_depth[m]) + 'cm_weighted_col_sm'

        for r in range(len(sensor_depths[0:n]) + 1):
            if r == 0:
                depth_weights[r] = (((sensor_depths[r+1] - sensor_depths[r])/2 + 
                                     sensor_depths[r])/Column_depth[m])
            elif ((r >= 0) and (r < len(sensor_depths[0:n])-1)):
                depth_weights[r] = ((sensor_depths[r+1] - sensor_depths[r-1])/2)/Column_depth[m] 
            else:
                depth_weights[r] = ((sensor_depths[r] - sensor_depths[r-1])/2)/Column_depth[m]
        print('depth weights =', depth_weights)

        for c in range(len(sensor_depths[0:n]) + 1):
            if c == 0:
                data[Weighted_column_names] = depth_weights[c] * data.iloc[:,c + 3]
            else:
                data[Weighted_column_names] =( data[Weighted_column_names] +
                                              depth_weights[c] * data.iloc[:,c + 3])

    return data


In [81]:
SM_data_weighted = column_weighted_sm(SM_data_cleaned, Column_sum_depth, sensor_depths)
SM_data_weighted.head()   

depth weights = [0.75 0.25 0.   0.   0.   0.  ]
depth weights = [0.075 0.075 0.2   0.15  0.25  0.   ]


,Station,Date & Time (CST),Precipitation (in),Soil Moisture 5cm (wfv),Soil Moisture 10cm (wfv),Soil Moisture 20cm (wfv),Soil Moisture 50cm (wfv),Soil Moisture 100cm (wfv),Soil Moisture 150cm (wfv),Precipitation_diff,Precipitation_downshift,Precipitation_upshift,10cm_weighted_col_sm,100cm_weighted_col_sm
0,bbc,2009-01-01 00:00:00,0.0,0.27,0.28,0.2,0.21,0.18,0.08,NaN,NaN,0.0,0.2725,0.15775
1,bbc,2009-01-01 01:00:00,0.0,0.27,0.28,0.2,0.22,0.18,0.08,0.0,0.0,0.0,0.2725,0.15925
2,bbc,2009-01-01 02:00:00,0.0,0.27,0.28,0.2,0.21,0.18,0.09,0.0,0.0,0.0,0.2725,0.15775
3,bbc,2009-01-01 03:00:00,0.0,0.27,0.28,0.2,0.21,0.18,0.09,0.0,0.0,0.0,0.2725,0.15775
4,bbc,2009-01-01 04:00:00,0.0,0.27,0.28,0.2,0.21,0.18,0.09,0.0,0.0,0.0,0.2725,0.15775


Missing some stations, no champaign station cci

## Partition storm events: 0 rain for 24 consecutive hours (eventually switch to 20/24 maybe)

ID indices for beginning and end of dry periods, subtract the difference and locate the dry periods 
greater than 24 hours and set those to be the cut off of the storm 

# assign last portion of data for each station that is less than consecutive dry hours to be False as we cannot no for certain if it meets the dry period cutoff definition

In [82]:
consecutive_dry_hours = 24
def precip_event_ID(data, hours):
    #min_num_dry_hours = np.floor(hours * 0.843) #(20/24)=0.8333333 same for 30/36
    Stations = data.Station.unique()
    data['Rain_ID'] = (data['Precipitation (in)'] > 0).astype(int)
    data['Rain_ID_lag'] = data['Rain_ID'].shift(1)
    for s in Stations:
        beginning_of_station = data.loc[data['Station'] == s].index[0]
        #data.iloc[beginning_of_station, 11] = np.NaN       

    data['Rain_ID_diff'] = data['Rain_ID'].sub(data['Rain_ID_lag'], axis = 0) #values of -1 in this column will always represent when a dry period started

    data['Dry_periods'] = data['Rain_ID_diff']
    data['Dry_periods'] = data['Dry_periods'].replace(1, -1) #now the days when dry periods start and end are 
    #represented as -1. Use index to grab the indices of these dates and subtract
    
    data['Dry_periods_overmin'] = False #set to false, find indices over 24 from dry period indices 
    #spacing and replace those with TRUE then select those with 
    print(Stations)
    for sn in Stations:
        station_data = data.loc[data['Station'] == sn]
        dry_period_indices = pd.Series(station_data.loc[station_data['Dry_periods'] == -1].index)
        dry_period_indices_spacing = dry_period_indices.diff()
        dry_period_indices_spacing_shifted = dry_period_indices_spacing.shift(-1) #aligning period lengths with the index they start with 
        dry_period_indices_spacing_shifted[-1:] = station_data.Dry_periods.index[-1:] - dry_period_indices[-1:]
        dry_over_consec_min = (dry_period_indices_spacing_shifted >= hours)
        dry_periods = pd.concat([dry_period_indices, dry_period_indices_spacing_shifted], axis = 1)
        dry_periods = dry_periods.loc[dry_over_consec_min]
        
        data.iloc[dry_periods[0], 18] = True
        
        beginning_of_station = data.loc[data['Station'] == sn].index[0]
        #assign last portion of data for each station that is less than consecutive dry hours to be 
        #False as we cannot no for certain if it meets the dry period cutoff definition
        data.iloc[beginning_of_station - consecutive_dry_hours:beginning_of_station + 1, 18] = False


    return data


In [83]:
WeightedSM_data_stormID = precip_event_ID(SM_data_weighted, consecutive_dry_hours)

['bbc' 'brw' 'bvl' 'dek' 'dxs' 'fai' 'fre' 'frm' 'icc' 'llc' 'oln' 'orr'
 'rnd' 'sfm' 'sni' 'stc' 'ste' 'cmi' 'mon' 'siu']


In [12]:
WeightedSM_data_stormID[0:50]

,Station,Date & Time (CST),Precipitation (in),Soil Moisture 5cm (wfv),Soil Moisture 10cm (wfv),Soil Moisture 20cm (wfv),Soil Moisture 50cm (wfv),Soil Moisture 100cm (wfv),Soil Moisture 150cm (wfv),Precipitation_diff,Precipitation_downshift,Precipitation_upshift,10cm_weighted_col_sm,100cm_weighted_col_sm,Rain_ID,Rain_ID_lag,Rain_ID_diff,Dry_periods,Dry_periods_overmin
0,bbc,2009-01-01 00:00:00,0.0,0.27,0.28,0.20,0.21,0.18,0.08,NaN,NaN,0.0,0.2725,0.15775,0,NaN,NaN,NaN,False
1,bbc,2009-01-01 01:00:00,0.0,0.27,0.28,0.20,0.22,0.18,0.08,0.0,0.0,0.0,0.2725,0.15925,0,0.0,0.0,0.0,False
2,bbc,2009-01-01 02:00:00,0.0,0.27,0.28,0.20,0.21,0.18,0.09,0.0,0.0,0.0,0.2725,0.15775,0,0.0,0.0,0.0,False
3,bbc,2009-01-01 03:00:00,0.0,0.27,0.28,0.20,0.21,0.18,0.09,0.0,0.0,0.0,0.2725,0.15775,0,0.0,0.0,0.0,False
4,bbc,2009-01-01 04:00:00,0.0,0.27,0.28,0.20,0.21,0.18,0.09,0.0,0.0,0.0,0.2725,0.15775,0,0.0,0.0,0.0,False
5,bbc,2009-01-01 05:00:00,0.0,0.27,0.28,0.20,0.21,0.18,0.09,0.0,0.0,0.0,0.2725,0.15775,0,0.0,0.0,0.0,False
6,bbc,2009-01-01 06:00:00,0.0,0.27,0.28,0.20,0.21,0.18,0.08,0.0,0.0,0.0,0.2725,0.15775,0,0.0,0.0,0.0,False
7,bbc,2009-01-01 07:00:00,0.0,0.27,0.28,0.20,0.21,0.18,0.08,0.0,0.0,0.0,0.2725,0.15775,0,0.0,0.0,0.0,False
8,bbc,2009-01-01 08:00:00,0.0,0.27,0.28,0.20,0.21,0.18,0.09,0.0,0.0,0.0,0.2725,0.15775,0,0.0,0.0,0.0,False
9,bbc,2009-01-01 09:00:00,0.0,0.27,0.28,0.20,0.22,0.18,0.09,0.0,0.0,0.0,0.2725,0.15925,0,0.0,0.0,0.0,False


'
""""""## select only data from April to October to avoid issues with the soil being frozen or snow fall

In [84]:
Apr_2_Oct_WeightedSM_data_stormID = WeightedSM_data_stormID.loc[
   (WeightedSM_data_stormID['Date & Time (CST)'].dt.month >= 4) & 
   (WeightedSM_data_stormID['Date & Time (CST)'].dt.month <= 10)]
Apr_2_Oct_WeightedSM_data_stormID = Apr_2_Oct_WeightedSM_data_stormID.reset_index(drop = True)


### select rows where Rain_ID_diff = -1 (beginning of dry period) and Dry_periods_over24 = True this is where the storm event ends
### Rain_ID_diff = 1 is beginning
select sm for timestep before beginning

variables to pull for each event (1 data point per evvent): sm-1, max_sm, average precip rate and peak precip, 

peak soil moisture values must bwe taken from the time the storm starts until the next event starts to accommodate ponding at the surface

## note that sometimes the first point in the dataset has rain and is taken as the start of an event, but we don't know if it is the true start and we don't have the prior sm should we throw these events out then? Yes, events are deleted

In [94]:
def Rain_events_data(data):
    Stations = data.Station.unique() 
    df_counter = 0

    #create new dataframe and asign data and station ID to it columnnames 
    #['Station', 'Avg_Precipitation_rate_in', 'Peak_Precipitation_rate_in', 
    #'Median_Precipitation_rate_in', 'Prior_sm_10', 'peak_sm_10' ,'Prior_sm_100', 'peak_sm_100']
    new_df = pd.DataFrame(columns = ['Station', 'Event_Duration', 'Avg_Precipitation_rate_in', 'Peak_Precipitation_rate_in', 
    'Median_Precipitation_rate_in', 'Total_Precipitation_in', 'Prior_sm_10', 'Peak_sm_10' ,
    'Prior_sm_100', 'Peak_sm_100'])

    full_prior_sm_10 = pd.Series()
    full_peak_sm_10 = pd.Series()
    full_prior_sm_100 = pd.Series()
    full_peak_sm_100 = pd.Series()
    full_Precip_peak_rate = pd.Series()
    full_Precip_avg_rate = pd.Series()
    full_Precip_med_rate = pd.Series()
    full_Precip_total = pd.Series()
    full_event_duration = pd.Series()
    station_id_series = pd.Series()

    for sn in Stations:
        print(sn)
        station_data = data.loc[data['Station'] == sn]
        first_event_beginning_index = pd.Series(station_data.loc[station_data['Rain_ID'] == 1].index[0])
        
        event_ending_indices = pd.Series(station_data.loc[(station_data['Rain_ID_diff'] == -1) & 
                                                (station_data['Dry_periods_overmin'] == True)].index)
        
        raining_begins_again = pd.Series(station_data.loc[station_data['Rain_ID'] == 1].index)
        start_of_second_storm_index = raining_begins_again.loc[
            raining_begins_again.astype(int) < event_ending_indices[1].astype(int)][-1:].values

        true_event_start = pd.Series(np.zeros(len(event_ending_indices))) 
        #there should be the same number for starts as finishes
        true_event_start[0] = first_event_beginning_index.iloc[0]

        event_duration = pd.Series(np.zeros(len(event_ending_indices)))
        prior_sm_10 = pd.Series(np.zeros(len(event_ending_indices)))
        peak_sm_10 = pd.Series(np.zeros(len(event_ending_indices)))
        prior_sm_100 = pd.Series(np.zeros(len(event_ending_indices)))
        peak_sm_100 = pd.Series(np.zeros(len(event_ending_indices)))
        Precip_peak_rate = pd.Series(np.zeros(len(event_ending_indices)))
        Precip_avg_rate = pd.Series(np.zeros(len(event_ending_indices)))
        Precip_med_rate = pd.Series(np.zeros(len(event_ending_indices)))
        Precip_total = pd.Series(np.zeros(len(event_ending_indices)))

        event_duration[0] = event_ending_indices[0].astype(int) - true_event_start[0].astype(int)
        if event_duration[0] == 0:
            print('event_duration[0] ==0?')
            print('event_ending_indices = ', event_ending_indices[0])
            print('true_event_start =', true_event_start[0])
            
        prior_sm_10[0] = data.iloc[true_event_start[0].astype(int)-1, 12]
        prior_sm_100[0] = data.iloc[true_event_start[0].astype(int)-1, 13]
        peak_sm_10[0] = data.iloc[true_event_start[0].astype(int):start_of_second_storm_index[0], 12].max()
        peak_sm_100[0] = data.iloc[true_event_start[0].astype(int):start_of_second_storm_index[0], 13].max()
        Precip_peak_rate[0] = data.iloc[true_event_start[0].astype(int):event_ending_indices[0].astype(int), 2].max()
        Precip_avg_rate[0] = data.iloc[true_event_start[0].astype(int):event_ending_indices[0].astype(int), 2].mean()
        Precip_med_rate[0] = data.iloc[true_event_start[0].astype(int):event_ending_indices[0].astype(int), 2].median()
        Precip_total[0] = data.iloc[true_event_start[0].astype(int):event_ending_indices[0].astype(int), 2].sum()              
        #pull out the true event starts
        print('event_end len =', len(event_ending_indices))



        for n in range(1,len(event_ending_indices)):
             #need to pull for last event, but sometimes the record doesn't have the next rain and
                # sometimes it does. if the next rain after the last identifiable event isn't in the 
                #record, need to throw out the last event as we don't know when the peak sm from the 
                # event is

                #ID if there is any rain after the event end in data
            if np.any(raining_begins_again > event_ending_indices[n]): 
                precip_after_last_event_end_index = raining_begins_again.loc[(raining_begins_again.astype(int) > event_ending_indices[n - 1].astype(int))].values[0]
                
                if (event_ending_indices[n].astype(int) - precip_after_last_event_end_index) < 0:
                    print('n', n)
                    print('ERROR: something wrong with event indices. Rain beginning after '
                    'last event end is after current end-backwards order of event end and beginning')
                    print('previous event end index',event_ending_indices[n - 1].astype(int))
                    print('current event ending', event_ending_indices[n].astype(int))
                    print('raining before current event', raining_begins_again.loc[
                        (raining_begins_again.astype(int) < event_ending_indices[n].astype(int))].values[-4:])
                    print('raining after previous event', raining_begins_again.loc[
                        (raining_begins_again.astype(int) > event_ending_indices[n - 1].astype(int))].values[0:4])
                    break
                
                true_event_start[n] = raining_begins_again.loc[
                    (raining_begins_again.astype(int) > event_ending_indices[n - 1].astype(int))
                                                                ].values[0]
                
                next_storm_start_index = raining_begins_again.loc[
                            raining_begins_again.astype(int) > 
                            event_ending_indices[n].astype(int)].values[0]
            
                #pull out variables between start and end of events and put into pd.series to 
                # assign to newdf below
                event_duration[n] = (event_ending_indices[n].astype(int) - 
                                     true_event_start[n].astype(int))
                if event_duration[n] == 0:
                    print('Event duration = 0?')
                    print('n', n)

                prior_sm_10[n] = data.iloc[true_event_start[n].astype(int)-1, 12]
                prior_sm_100[n] = data.iloc[true_event_start[n].astype(int)-1, 13]
                peak_sm_10[n] = data.iloc[true_event_start[n].astype(int):
                                          next_storm_start_index, 12].max() #put in this "next_storm_start_index" where "event_ending_indices[n].astype(int)"" to account for pooling 
                peak_sm_100[n] = data.iloc[true_event_start[n].astype(int):
                                           next_storm_start_index, 13].max() #put in this to account for pooling next_storm_start_index
                Precip_peak_rate[n] = data.iloc[true_event_start[n].astype(int):
                                                event_ending_indices[n].astype(int), 2].max()
                Precip_avg_rate[n] = data.iloc[true_event_start[n].astype(int):
                                               event_ending_indices[n].astype(int), 2].mean()
                Precip_med_rate[n] = data.iloc[true_event_start[n].astype(int):
                                               event_ending_indices[n].astype(int), 2].median()
                Precip_total[n] = data.iloc[true_event_start[n].astype(int):
                                            event_ending_indices[n].astype(int), 2].sum()
                
                if peak_sm_10[n] - prior_sm_10[n] < 0:
                    print('Soil Moisture Change in 10cm is negative!')
                    print('calc value = ',peak_sm_10[n] - prior_sm_10[n])
                    print('n', n)
                    print('station', sn)
                    print('current event start index', true_event_start[n].astype(int))
                    print('current event end index', event_ending_indices[n].astype(int))
                    print('next event start index', next_storm_start_index)

                if peak_sm_100[n] - prior_sm_100[n] < 0:
                    print('Soil Moisture Change in 100cm is negative!')
                    print('calc value =', peak_sm_100[n] - prior_sm_100[n])
                    print('n', n)
                    print('station', sn)
                    print('current event start index', true_event_start[n].astype(int))
                    print('current event end index', event_ending_indices[n].astype(int))
                    print('next event start index', next_storm_start_index)

            else:
                #if there is no rain after the last event end, need to throw out event as we cannot 
                # ID peak SM from start of last event to next rain, there is no next rain
                print('welp, throw out')
                print('event end index to check against',event_ending_indices[n] )
                print('rain after current event ends (next event)',raining_begins_again.loc[
                        raining_begins_again.astype(int) > 
                        event_ending_indices[n].astype(int)].values)
                print('event ending indices before cutting off last', event_ending_indices[-4:])
                print('event starting indices before cutting off last', true_event_start[-4:])
                event_ending_indices = event_ending_indices[:-1]
                true_event_start = true_event_start[:-1]
                print('event ending index after cutting off last', event_ending_indices[-4:])
                print('event starting indices after cutting off last', true_event_start[-4:])
                break

            #throw out first event if it begins at the beginning of the time series as we do not 
            # know when it actually started
        if first_event_beginning_index.iloc[0] == 0:
            true_event_start = true_event_start[1:]
            event_ending_indices = event_ending_indices[1:]
            event_duration = event_duration[1:]
            prior_sm_10 = prior_sm_10[1:]
            prior_sm_100 = prior_sm_100[1:]
            peak_sm_10 = peak_sm_10[1:]
            peak_sm_100 = peak_sm_100[1:]
            Precip_avg_rate = Precip_avg_rate[1:]
            Precip_med_rate = Precip_med_rate[1:]
            Precip_total =  Precip_total[1:]
            Precip_peak_rate = Precip_peak_rate[1:]

        """""
        #plotting below to check event ID

        #last_precip_marker = station_data.loc[station_data['Rain_ID']==1].index[-1:]
        #print('index of last precipitation', last_precip_marker)
        #last_datapoint = station_data.Rain_ID.index[-1:]
        #print('length of station data',last_datapoint )
        precip = station_data['Precipitation (in)'][:600]
        sm_10 = station_data['10cm_weighted_col_sm'][:600]
        fig, ax = plt.subplots(nrows =1, ncols =2, figsize = (12,4))
        ax[0].plot(precip, color = 'cornflowerblue')
        #x = event_ending_indices[len(event_ending_indices)-1]
        #ax.axvline(x=x, color ='red')
        
        ax[0].set_ylabel('Precipitation (in)', fontsize = 14)
        ax[0].set_xlabel('timeseries location', fontsize = 14)
        ax[0].set_xticks([])

        ax[1].plot(sm_10, color = 'peru')
        #ax[1].set_title('Last four events for station %s' % sn, fontsize = 19)
        ax[1].set_ylabel('Weighted 10cm Soil Moisture (wfv)', fontsize = 14)
        ax[1].set_xlabel('timeseries location', fontsize = 14)
        ax[1].set_xticks([])

        fig.suptitle('first 6 Events for Station %s' % sn, fontsize = 19)
        event_number = 6

        Last_4_start_times = true_event_start[:event_number].tolist()
        print('true event start index',Last_4_start_times)
        Last_4_end_times = event_ending_indices[:event_number].tolist()
        print('true event end indices', Last_4_end_times)
        
        for v in range(event_number):
            print('length of the initial events =',(Last_4_end_times[v] - Last_4_start_times[v]))
            ax[0].axvline(x = Last_4_start_times[v], color = 'green', linestyle = '-')
            ax[0].axvline(x = Last_4_end_times[v], color = 'red', linestyle = '-.')
            ax[1].axvline(x = Last_4_start_times[v], color = 'green', linestyle = '-')
            ax[1].axvline(x = Last_4_end_times[v], color = 'red', linestyle = '-.')
        
        plt.show()
        """
        
        station_id_series_current_loop = pd.Series(np.repeat(sn, len(event_ending_indices)))

        if df_counter == 0:
            #for first iteration with first station
            full_event_duration = event_duration
            full_prior_sm_10 = prior_sm_10
            full_peak_sm_10 = peak_sm_10
            full_prior_sm_100 = prior_sm_100
            full_peak_sm_100 = peak_sm_100
            full_Precip_peak_rate = Precip_peak_rate
            full_Precip_avg_rate = Precip_avg_rate
            full_Precip_med_rate = Precip_med_rate
            full_Precip_total = Precip_total
            station_id_series = station_id_series_current_loop
        else:
            #for after first station, concat vertically
            full_event_duration = pd.concat([full_event_duration, event_duration], axis = 0)
            full_prior_sm_10 = pd.concat([full_prior_sm_10, prior_sm_10], axis = 0)
            full_peak_sm_10 = pd.concat([full_peak_sm_10, peak_sm_10], axis = 0)
            full_prior_sm_100 = pd.concat([full_prior_sm_100, prior_sm_100], axis = 0)
            full_peak_sm_100 = pd.concat([full_peak_sm_100, peak_sm_100], axis = 0)
            full_Precip_peak_rate = pd.concat([full_Precip_peak_rate, Precip_peak_rate], axis = 0)
            full_Precip_avg_rate = pd.concat([full_Precip_avg_rate, Precip_avg_rate], axis = 0)
            full_Precip_med_rate = pd.concat([full_Precip_med_rate, Precip_med_rate], axis = 0)
            full_Precip_total = pd.concat([full_Precip_total, Precip_total], axis = 0)
            station_id_series = pd.concat([station_id_series, station_id_series_current_loop], axis = 0)

        df_counter += 1
    print(len(full_peak_sm_10))
    print(len(full_Precip_avg_rate))

    station_id_series, full_event_duration ,full_Precip_avg_rate, full_Precip_med_rate, full_Precip_peak_rate, full_Precip_total, full_prior_sm_10, full_peak_sm_10, full_prior_sm_100, full_peak_sm_100 = [
        s.reset_index(drop = True) for s in [station_id_series, full_event_duration, 
                                             full_Precip_avg_rate, full_Precip_med_rate, 
                                             full_Precip_peak_rate, full_Precip_total, 
                                             full_prior_sm_10, full_peak_sm_10, 
                                             full_prior_sm_100, full_peak_sm_100]
    ]

    #['Station', 'Event_Duration', 'Avg_Precipitation_rate_in', 'Peak_Precipitation_rate_in', 
    #'Median_Precipitation_rate_in', 'Prior_sm_10', 'Peak_sm_10' ,'Prior_sm_100', 'Peak_sm_100']
    new_df = new_df.assign(Station = station_id_series,
                           Event_Duration = full_event_duration,
                           Avg_Precipitation_rate_in = full_Precip_avg_rate,
                           Peak_Precipitation_rate_in = full_Precip_peak_rate,
                           Median_Precipitation_rate_in = full_Precip_med_rate,
                           Total_Precipitation_in = full_Precip_total,
                           Prior_sm_10 = full_prior_sm_10,
                           Peak_sm_10 = full_peak_sm_10,
                           Prior_sm_100 = full_prior_sm_100,
                           Peak_sm_100 = full_peak_sm_100)
    
    #delete rows with Nan values because the kde gaussian function cannot handle the nans 
    #and keeping one column in a row for data but not another (when only the sm or rain gauge is out
    #for a row) makes mapping values to eachother impossbile so mutual information cannot be 
    # appropriately informed need to be able to calc. p(xi, yi)

    new_df = new_df.dropna(how = 'any')
    new_df = new_df.reset_index(drop = True)
    

    return new_df



In [95]:
#Event_vars_df = Rain_events_data(WeightedSM_data_stormID) 
Apr_2_oct_Event_vars_df = Rain_events_data(Apr_2_Oct_WeightedSM_data_stormID)

bbc
event_end len = 734
Soil Moisture Change in 10cm is negative!
calc value =  -0.0005000000000000004
n 138
station bbc
current event start index 19101
current event end index 19103
next event start index 19215
Soil Moisture Change in 100cm is negative!
calc value = -7.499999999999174e-05
n 177
station bbc
current event start index 24326
current event end index 24330
next event start index 24357
Soil Moisture Change in 10cm is negative!
calc value =  -0.0007499999999999729
n 180
station bbc
current event start index 24482
current event end index 24484
next event start index 24508
Soil Moisture Change in 10cm is negative!
calc value =  -0.0005000000000000004
n 181
station bbc
current event start index 24508
current event end index 24546
next event start index 24687
Soil Moisture Change in 10cm is negative!
calc value =  -0.0010000000000000009
n 208
station bbc
current event start index 27104
current event end index 27105
next event start index 27141
Soil Moisture Change in 100cm is neg

In [99]:
Apr_2_oct_Event_vars_df.loc[Apr_2_oct_Event_vars_df['Event_Duration'] == 0]

,Station,Event_Duration,Avg_Precipitation_rate_in,Peak_Precipitation_rate_in,Median_Precipitation_rate_in,Total_Precipitation_in,Prior_sm_10,Peak_sm_10,Prior_sm_100,Peak_sm_100
1297,bvl,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1961,dek,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3358,fai,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3999,fre,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5341,icc,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6038,llc,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6686,oln,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7347,orr,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8013,rnd,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8690,sfm,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Calculate the difference in soil moisture values the delta(sm) we are interested in how precipitaiton changes the sm recharge

In [87]:
#this function assumes that the data is in the same structure I created earlier where the first sm 
#data column is 'Prior_sm_##' and there after alternates with its corresponding 'Peak_sm_##' to the 
#last column
def delta_sm(data):
    first_prior_sm_col = data.columns.str.contains('Prior_sm_').argmax() #.argmax returns the column index
    sm_cols = data.columns[first_prior_sm_col:] #pulls out all columns from first sm col to last
    ptot_col = data.columns[data.columns.str.contains('Total').argmax()]
    for col in sm_cols:
        if col.startswith('Prior_sm_'):
            depth = col.split('_')[-1]
            peak_col = f'Peak_sm_{depth}'
            delta_col = f'Delta_sm_{depth}'

            #check if the corresponding peak sm column exists to calculate the delta

            if peak_col in data.columns:
                data[delta_col] = data[peak_col] - data[col]
                data[delta_col+'_perPtot'] = data[delta_col]/data[ptot_col]

    
    return  data

In [88]:
Apr_2_oct_Event_vars_df = delta_sm(data = Apr_2_oct_Event_vars_df)

In [97]:
Apr_2_oct_Event_vars_df.loc[Apr_2_oct_Event_vars_df['Event_Duration']== 0]

,Station,Event_Duration,Avg_Precipitation_rate_in,Peak_Precipitation_rate_in,Median_Precipitation_rate_in,Total_Precipitation_in,Prior_sm_10,Peak_sm_10,Prior_sm_100,Peak_sm_100
1297,bvl,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1961,dek,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3358,fai,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3999,fre,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5341,icc,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6038,llc,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6686,oln,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7347,orr,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8013,rnd,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8690,sfm,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### delete the events with delta sm < -0.01

 then change the others which are negative but -0.00## since this is beyond sensor precision

In [62]:
#delete the events with < -0.01
Apr_2_oct_Event_vars_df.drop(Apr_2_oct_Event_vars_df[Apr_2_oct_Event_vars_df['Delta_sm_100'] < -0.01].index, inplace = True)

Apr_2_oct_Event_vars_df.drop(Apr_2_oct_Event_vars_df[Apr_2_oct_Event_vars_df['Delta_sm_10'] < -0.01].index, inplace = True)

In [63]:
Apr_2_oct_Event_vars_df.loc[Apr_2_oct_Event_vars_df['Delta_sm_10'] < 0.0, ('Delta_sm_10', 'Delta_sm_10_perPtot')] = 0
Apr_2_oct_Event_vars_df.loc[Apr_2_oct_Event_vars_df['Delta_sm_100'] < 0.0, ('Delta_sm_100', 'Delta_sm_100_perPtot')] = 0



### Normalize data (rescale to [0,1]) with min-max scaling

## x-xmin/(xmax - xmin)
this will not change the data shape only it's range and thus it maintains it underlying behaviors and relationships in paired data


this will also help with the distance issue in the curse of dimensionality for the joint distribution

In [66]:
def min_max_normalization(data_multi_df):
    stations = data_multi_df.Station.unique()
    normalized_data_multi_df =[]
    min_max_mean_med_cols = []
    col_to_normalize_list = data_multi_df.columns[
        data_multi_df.columns.str.contains('Precipitation|Delta|Prior')].tolist()
    print(col_to_normalize_list)
    for s in stations:
        station_df = data_multi_df.loc[data_multi_df['Station'] == s].copy()
        station_min_max_mean_med_cols = []
        for col in col_to_normalize_list:
            station_df_col_min = station_df[col].min()
            station_df_col_max = station_df[col].max()
            station_df_col_mean = station_df[col].mean()
            station_df_col_median = station_df[col].median()
        
            station_df.loc[:,col] = ((station_df[col] - station_df_col_min) / 
                               (station_df_col_max - station_df_col_min))
            
            min_max_mean_med_cols.append({
                                    'Station': s,
                                    'Variable': col, 
                                    'Min': station_df_col_min, 
                                    'Max': station_df_col_max, 
                                    'Mean': station_df_col_mean, 
                                    'Median': station_df_col_median})
        

        normalized_data_multi_df.append(station_df)
    normalized_data_multi_df = pd.concat(normalized_data_multi_df, axis = 0).reset_index(drop = True)
    min_max_mean_med_df = pd.DataFrame(min_max_mean_med_cols)
        
    return normalized_data_multi_df, min_max_mean_med_df


In [67]:
Apr_2_Oct_Station_events_df_normalized, min_max_data_df = min_max_normalization(Apr_2_oct_Event_vars_df)

['Avg_Precipitation_rate_in', 'Peak_Precipitation_rate_in', 'Median_Precipitation_rate_in', 'Total_Precipitation_in', 'Prior_sm_10', 'Prior_sm_100', 'Delta_sm_10', 'Delta_sm_10_perPtot', 'Delta_sm_100', 'Delta_sm_100_perPtot']


In [46]:
min_max_data_df.loc[min_max_data_df['Station']=='bbc']

,Station,Variable,Min,Max,Mean,Median
0,bbc,Avg_Precipitation_rate_in,0.000242,1.116000,0.058715,0.024588
1,bbc,Peak_Precipitation_rate_in,0.002000,2.417000,0.239094,0.123000
2,bbc,Median_Precipitation_rate_in,0.000000,1.116000,0.038170,0.005000
3,bbc,Total_Precipitation_in,0.002000,5.406000,0.627645,0.295000
4,bbc,Prior_sm_10,0.041750,0.295250,0.218324,0.239750
5,bbc,Prior_sm_100,0.041925,0.180275,0.133197,0.138900
6,bbc,Delta_sm_10,0.000000,0.188750,0.028409,0.015750
7,bbc,Delta_sm_10_perPtot,0.000000,2.625000,0.104978,0.044444
8,bbc,Delta_sm_100,0.000000,0.066575,0.007977,0.003525
9,bbc,Delta_sm_100_perPtot,0.000000,2.325000,0.043478,0.011193


In [23]:
"""Apr_2_Oct_Station_events_df_normalized.loc[Apr_2_Oct_Station_events_df_normalized['Station']=='siu'].boxplot(column =['Avg_Precipitation_rate_in',
                                                        'Median_Precipitation_rate_in', 
                                                        'Peak_Precipitation_rate_in', 
                                                        'Total_Precipitation_in',
                                                        'Delta_sm_10',
                                                        'Delta_sm_100'])
plt.xticks(rotation = 90)"""

"Apr_2_Oct_Station_events_df_normalized.loc[Apr_2_Oct_Station_events_df_normalized['Station']=='siu'].boxplot(column =['Avg_Precipitation_rate_in',\n                                                        'Median_Precipitation_rate_in', \n                                                        'Peak_Precipitation_rate_in', \n                                                        'Total_Precipitation_in',\n                                                        'Delta_sm_10',\n                                                        'Delta_sm_100'])\nplt.xticks(rotation = 90)"

### PDF estimation with KDE, binwidth method options are scott or silvermann, using scott's method as it is more robust for non-normally distributed data with skewness, which is precip data
https://ntnuopen.ntnu.no/ntnu-xmlui/bitstream/handle/11250/258553/348940_FULLTEXT01.pdf?sequence=1&isAllowed=y
https://www.umiacs.umd.edu/labs/cvl/pirl/vikas/publications/optimal_bw_slides.pdf

Goodwell and Kumar used Silverman it seems based on their citations of Silverman

set up pdf_generation function to be expandable to any number of columns (1st time through we are only using sm for 10 cm and 100cm)
also set up to be expandable to the number of stations incase we were to give this more station data from other states

In [68]:
#adjust data within function to be dictionary of multiple dfs
def Layer_dfs(data):
    data_columns = data.columns[1:]
    Stations = data.Station.unique()
    multi_dfs = {} #make dictionary of multiple dfs
    for d in Stations:
        print(f'Creating df for {d}')
        #initialize empty single df
        single_df = pd.DataFrame()
        for col in data_columns:
            single_df[col] = data.loc[data.Station == d, col]
        #store single df in dictionary of multiple dfs

        multi_dfs[d] = single_df
    return multi_dfs
#station_dfs = Layer_dfs(Event_vars_df)


In [69]:
Apr_2_Oct_station_dfs = Layer_dfs(Apr_2_Oct_Station_events_df_normalized)

Creating df for bbc
Creating df for brw
Creating df for bvl
Creating df for dek
Creating df for dxs
Creating df for fai
Creating df for fre
Creating df for frm
Creating df for icc
Creating df for llc
Creating df for oln
Creating df for orr
Creating df for rnd
Creating df for sfm
Creating df for sni
Creating df for stc
Creating df for ste
Creating df for cmi
Creating df for mon
Creating df for siu


In [75]:
#save data and kde df
import pickle

with open(r"/data/keeling/a/cstraus2/Phd_research/Apr_2_Oct_Station_events_data_multidf_negdeltas_del_normalized_delta_perPtot.pkl", "wb") as f:
    pickle.dump(Apr_2_Oct_station_dfs, f)